<a href="https://colab.research.google.com/github/24202795-oss/data-science-toolkit/blob/main/Tutorial6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler


df_salary = pd.read_csv("/content/drive/MyDrive/salary.csv")
df_xtra = pd.read_csv("/content/drive/MyDrive/xtra_info.csv")


print("Salary table size: ", df_salary.shape)
print("Extra table size: ", df_xtra.shape)


display(df_salary.head())
display(df_xtra.head())

Salary table size:  (28180, 19)
Extra table size:  (13404, 4)


,Unnamed: 0,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,0,2021-04-27 11:02:09.743,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,55000,0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,1,2021-04-27 11:02:21.562,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,54600,4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,2,2021-04-27 11:02:38.125,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,34000,NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,3,2021-04-27 11:02:40.643,25-34,Nonprofits,Program Manager,NaN,62000,3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4,2021-04-27 11:02:41.793,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,60000,7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White


,USERID,CGPA,Salary_month_USD,Special Bonus_GBP
0,8683,2.990627,4583.333333,2437.0
1,3880,2.288131,5166.666667,740.0
2,8898,3.903324,5000.000000,3806.0
3,4441,2.470852,9333.333333,4691.0
4,7537,3.559457,3750.000000,4580.0


In [10]:
short_names = {
    df_salary.columns[0]:"USERID",
    df_salary.columns[6]:"Annual_Salary",
    df_salary.columns[7]:"Extra_Compensation",
    df_salary.columns[8]:"Currency",
    df_salary.columns[11]:"Country",
    df_salary.columns[13]:"City"

}

df_salary = df_salary.rename(columns=short_names)
df_salary["USERID"] = pd.to_numeric(df_salary ["USERID"], errors="coerce")
df_salary["Annual_Salary"] = pd.to_numeric(
    df_salary ["Annual_Salary"].astype(str).str.replace(",", "", regex=False), errors="coerce")

df_xtra["USERID"]= pd.to_numeric(df_xtra["USERID"], errors="coerce")
display(df_salary [["USERID", "Annual_Salary", "Currency", "Country", "City"]].head())

,USERID,Annual_Salary,Currency,Country,City
0,0,55000,USD,United States,Boston
1,1,54600,GBP,United Kingdom,Cambridge
2,2,34000,USD,US,Chattanooga
3,3,62000,USD,USA,Milwaukee
4,4,60000,USD,US,Greenville


In [12]:

print("Extra rows:", len(df_xtra))
print("Unique extra USERIDS:", df_xtra["USERID"].nunique())

print("Repeated USERID rows:", df_xtra.duplicated(subset="USERID").sum())

df_xtra_unique = df_xtra.drop_duplicates(subset="USERID", keep="first").copy()
print("After removing repeated IDs:", len(df_xtra_unique))
assert not df_xtra_unique["USERID"].duplicated().any()

Extra rows: 13404
Unique extra USERIDS: 8442
Repeated USERID rows: 4962
After removing repeated IDs: 8442


In [13]:
df_integrated=pd.merge(
    df_salary,
    df_xtra_unique,
    on="USERID",
    how="left",
    validate="one_to_one"
    )

print("Salary rows before merge:", len(df_salary))

print("Salary rows after merge:", len(df_integrated))

print("Rows with extra information:", df_integrated["CGPA"].notna().sum())

assert len(df_integrated) == len(df_salary)



display(df_integrated[["USERID", "Annual_Salary", "Currency","CGPA", "Salary_month_USD", "Special Bonus_GBP"]].head(8))

Salary rows before merge: 28180
Salary rows after merge: 28180
Rows with extra information: 8442


,USERID,Annual_Salary,Currency,CGPA,Salary_month_USD,Special Bonus_GBP
0,0,55000,USD,NaN,NaN,NaN
1,1,54600,GBP,2.695615,4250.000000,125.0
2,2,34000,USD,NaN,NaN,NaN
3,3,62000,USD,NaN,NaN,NaN
4,4,60000,USD,2.689383,12500.000000,1133.0
5,5,62000,USD,NaN,NaN,NaN
6,6,33000,USD,3.541300,6333.333333,878.0
7,7,50000,USD,NaN,NaN,NaN


In [15]:

comparison = df_integrated.loc[
    (df_integrated["Currency"] == "USD") &
    df_integrated["Salary_month_USD"].notna(),["USERID", "Annual_Salary", "Salary_month_USD"]].copy()

comparison["Calculated_Monthly_USD"] = comparison["Annual_Salary"] / 12
rough_match=(
    comparison["Salary_month_USD"] - comparison["Calculated_Monthly_USD"]).abs() < 0.01

print("Number of comparable records:", len(comparison))
print("Matching monthly amounts:", rough_match.sum())
print("Nonmatching monthly amounts:", (~rough_match).sum())

display(comparison.head())

Number of comparable records: 7063
Matching monthly amounts: 55
Nonmatching monthly amounts: 7008


,USERID,Annual_Salary,Salary_month_USD,Calculated_Monthly_USD
4,4,60000,12500.000000,5000.000000
6,6,33000,6333.333333,2750.000000
9,9,45000,12500.000000,3750.000000
10,10,47500,4583.333333,3958.333333
12,12,100000,12083.333330,8333.333333


In [16]:
# A genuinely derivable feature is yearly salary divided by 12.

df_integrated["Annual_Salary_Div_12"] = df_integrated["Annual_Salary"] / 12

# Remove that calculated column because we can recreate it whenever needed.

df_integrated = df_integrated.drop(columns=["Annual_Salary_Div_12"])

# Confirm it has been removed.

print("Derived example column removed:",
      "Annual_Salary_Div_12" not in df_integrated.columns)

Derived example column removed: True


In [18]:
#Classroom exchange-rate assumption, not a live exchange rate.

GBP_TO_USD=1.30

#Create a new column containing the converted bonus in USD.

df_integrated["Special_Bonus_USD"] =(
    pd.to_numeric(df_integrated["Special Bonus_GBP"], errors="coerce")* GBP_TO_USD
    )

#errors=coerce means "If a value cannot be converted into a number, replace it with NaN (missing value) instead of showing an error

#Drop the old GBP bonus column now that we have a USD version.

df_integrated = df_integrated.drop(columns=["Special Bonus_GBP"])

# Display the converted bonus for a few matching records.

display(df_integrated.loc[
    df_integrated["Special_Bonus_USD"].notna(),
     ["USERID", "Special_Bonus_USD"]].head())

,USERID,Special_Bonus_USD
1,1,162.5
4,4,1472.9
6,6,1141.4
9,9,6277.7
10,10,1016.6


In [19]:
# Select only salaries reported in US dollars.

df_usd = df_integrated.loc[df_integrated["Currency"] == "USD"].copy()

# Keep only sensible, numeric, positive salaries

df_usd=df_usd.loc[
    df_usd["Annual_Salary"].notna() & (df_usd["Annual_Salary"] > 0)
    ].copy()

# Print the size and sample annual salary values.

print("USD salary records:", len(df_usd))
display(df_usd[["Annual_Salary"]].head())

USD salary records: 23441


,Annual_Salary
0,55000
2,34000
3,62000
4,60000
5,62000


In [20]:
#Create a MinMaxScaler, which learns the smallest and largest salaries.
minmax=MinMaxScaler()

#fit transform learns the range and scales the column to 0-1.
df_usd["Salary_MinMax"]=minmax.fit_transform(df_usd[["Annual_Salary"]]).ravel()

#ravel converts the result from a two-dimensional array into a one-dimensional array
#from this [[0.0], [0.5], [1.0]] to this [0.0, 0.5, 1.0]

# Compare the original salary with its scaled value.
display(df_usd[["Annual_Salary", "Salary_MinMax"]].head(10))
print("Minimum scaled salary:", df_usd["Salary_MinMax"].min())
print("Maximum scaled salary:", df_usd["Salary_MinMax"].max())

,Annual_Salary,Salary_MinMax
0,55000,0.000539
2,34000,0.000333
3,62000,0.000608
4,60000,0.000588
5,62000,0.000608
6,33000,0.000324
7,50000,0.000490
8,112000,0.001098
9,45000,0.000441
10,47500,0.000466


Minimum scaled salary: 0.0
Maximum scaled salary: 1.0


In [21]:
#Create a StandardScaler
zscore_scaler=StandardScaler()

# Fit the scaler using the annual USD salaries and transform them.
df_usd["Salary_Zscore"] = zscore_scaler.fit_transform(
    df_usd[["Annual_Salary"]]).ravel()

# Compare the original salary with both scaling results.
display(df_usd[["Annual_Salary", "Salary_MinMax", "Salary_Zscore"]].head(10))

# Calculate the average of the standardized salaries
print("Mean:", df_usd["Salary_Zscore"].mean())

# Calculate how spread out the standardized salaries are
print("Standard deviation:", df_usd["Salary_Zscore"].std())

,Annual_Salary,Salary_MinMax,Salary_Zscore
0,55000,0.000539,-0.058724
2,34000,0.000333,-0.086545
3,62000,0.000608,-0.049450
4,60000,0.000588,-0.052100
5,62000,0.000608,-0.049450
6,33000,0.000324,-0.087870
7,50000,0.000490,-0.065348
8,112000,0.001098,0.016792
9,45000,0.000441,-0.071972
10,47500,0.000466,-0.068660


Mean: -9.699828311216759e-18
Standard deviation: 1.0000213308305577


In [22]:
# Remove rows with missing/empty city names.
# Remove rows with missing city names
df_city=df_usd.dropna(subset=["City"])

# groupby collects records in the same country/city combination.
# agg calculates different summary statistics for each group.

# This code groups employees by their country and city, then calculates salary statistics for each location.
# It puts rows belonging to the same country and city into groups.
# For example, all three New York salaries are grouped together.
# as_index=False simply keeps Country and City as normal columns in the resulting table.

city_summary=df_city.groupby (["Country", "City"], as_index=False).agg(
    Number_of_Records=("Annual_Salary", "count"), # How many reported salaries?
    Mean_Salary_USD=("Annual_Salary", "mean"), # Average annual salary
    Median_Salary_USD=("Annual_Salary", "median"), # Middle annual salary
    Minimum_Salary_USD=("Annual_Salary", "min"), # Smallest annual salary
    Maximum_Salary_USD=("Annual_Salary", "max") # Largest annual salary
    )

# Display a sample of the grouped results.
display(city_summary.head(10))

,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
0,U.S.,indianapolis,1,42500.0,42500.0,42500,42500
1,US,Philadelphia,1,86000.0,86000.0,86000,86000
2,United States,South Bend,1,69000.0,69000.0,69000,69000
3,Africa,Africa,1,90000.0,90000.0,90000,90000
4,America,Ann Arbor,1,34000.0,34000.0,34000,34000
5,America,Atlanta,1,45000.0,45000.0,45000,45000
6,America,Boston,1,194000.0,194000.0,194000,194000
7,America,Chicago,1,70.0,70.0,70,70
8,America,Columbia,1,75000.0,75000.0,75000,75000
9,America,Houston,1,20488.0,20488.0,20488,20488


In [23]:
#Sort all groups by average annual salary, highest to lowest.

all_cities_sorted=city_summary.sort_values(
    by="Mean_Salary_USD", ascending=False
    )

#Show the ten cities with the highest sample mean salary.
print("Top 10 city/country groups (all sizes):")
display(all_cities_sorted.head(10))

Top 10 city/country groups (all sizes):


,Country,City,Number_of_Records,Mean_Salary_USD,Median_Salary_USD,Minimum_Salary_USD,Maximum_Salary_USD
62,Colombia,Bogota,2,51012500.0,51012500.0,25000,102000000
64,Colombia,Cali,2,26106000.0,26106000.0,12000,52200000
6924,usa,Rice,1,10000000.0,10000000.0,10000000,10000000
6609,Usa,Pinebrook,1,5000044.0,5000044.0,5000044,5000044
59,China,Hong Kong,1,5000000.0,5000000.0,5000000,5000000
2524,USA,Newyork,1,2600000.0,2600000.0,2600000,2600000
175,Taiwan,Milwaukee,1,2000000.0,2000000.0,2000000,2000000
102,India,Banglore,1,1000000.0,1000000.0,1000000,1000000
3639,United Stated,Spokane,1,832000.0,832000.0,832000,832000
430,U.S.,Salina,1,734400.0,734400.0,734400,734400
